<a href="https://colab.research.google.com/github/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA/blob/main/notebooks/60_integracion/60_tablas_maestras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 60 · Tablas maestras: grilla de 2 km con variables y panel municipio × año

**Autor:** Daniel

**Propósito:** unir en dos tablas las variables de las cuatro partes, para el análisis integrador (cuadernos 61, 62 y 63).

**Entradas:**
- `datos/comunes/grilla_2km.gpkg` (cuaderno 02, DVC).
- Capas por celda en `datos/comunes/`, una fila por celda y la columna `id_celda`. Se agregan solas todas las que terminan en `_celda.csv`:
  - `focos_celda.csv` (cuaderno 22, Michelle).
  - `lluvia_celda.csv` (cuaderno 12, Michelle).
  - `quema_celda.csv` (cuaderno 24, Jonatan).
  - `agro_celda.csv` (cuaderno 30, Daniel).
  - `suelo_celda.csv` (cuaderno 42, Katalina).
- Tablas municipales de MapBiomas: `modulos/expansion/datos/mapbiomas_crudos/` (Git).
- EVA: `modulos/expansion/datos/eva/eva_altillanura_anual.csv` (Git).
- `datos/comunes/firms_mensual_municipio.csv`, `chirps_mensual_municipio.csv`, `oni_mensual.csv` y `altillanura_municipios.csv` (cuadernos 01, 03, 10 y 11).

**Salidas:**
- `datos/comunes/grilla_2km_variables.csv` (DVC): una fila por celda con todas las capas.
- `datos/comunes/panel_municipio_anio.csv`: una fila por municipio y año, 1985–2025.
- `datos/comunes/panel_municipio_mes.csv`: una fila por municipio y mes, 1981–2025.

**Cómo se actualiza:** cuando alguien sube una capa nueva por celda, se vuelve a correr este cuaderno y se versiona otra vez `grilla_2km_variables.csv` con DVC.

**Convenciones del panel:**
- Las variables que también tiene el panel del módulo de fuego y expansión (Jonatan) usan los mismos nombres (`area_total_ha`, `agro_pct`, `d_agro_pct`, `focos_seca_total`, `lluvia_seca_mm`…). Las de quema y pérdida de vegetación están en ese panel.
- La temporada seca de un año *t* va de diciembre de *t − 1* a marzo de *t*.
- Los focos de FIRMS son Terra + Aqua desde enero de 2003; antes de 2003 quedan vacíos.
- Un año o una temporada solo se calcula si tiene todos sus meses.

## Paso 1. Rutas y librerías

In [1]:
import os, sys, re, glob, subprocess
EN_COLAB = 'google.colab' in sys.modules
REPO = 'https://github.com/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA.git'

if EN_COLAB:
    RAIZ = '/content/Proyecto-ALTILLANURA'
    if not os.path.exists(RAIZ):
        subprocess.run(['git', 'clone', '-q', REPO, RAIZ], check=True)
else:
    RAIZ = os.path.abspath(os.path.join('..', '..'))

COMUNES = os.path.join(RAIZ, 'datos', 'comunes')
EXPANSION = os.path.join(RAIZ, 'modulos', 'expansion')

def traer(ruta):
    if os.path.exists(os.path.join(RAIZ, ruta)):
        return
    if EN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dvc[gs]'], check=True)
    subprocess.run([sys.executable, '-m', 'dvc', 'pull', '-q', ruta], cwd=RAIZ, check=True)
    print('Bajado de DVC:', ruta)

import numpy as np
import pandas as pd
import geopandas as gpd

print('Raíz del repo:', RAIZ)

Raíz del repo: c:\Users\mipip\Downloads\altillanura_proyecto_\altillanura_proyecto


## Paso 2. Grilla con todas las capas por celda
Se toma cada archivo `datos/comunes/*_celda.csv` (o su puntero `.dvc`). Solo se une si tiene `id_celda` y una fila por celda. Las columnas que ya trae la grilla (municipio, área y centroide) se toman de la grilla; si otra columna se repite entre capas, se le agrega el nombre de la capa al final.

In [2]:
traer('datos/comunes/grilla_2km.gpkg')
grilla = gpd.read_file(os.path.join(COMUNES, 'grilla_2km.gpkg'))
tabla = pd.DataFrame(grilla.drop(columns='geometry'))
DE_LA_GRILLA = list(tabla.columns)

ESPERADAS = {
    'focos_celda.csv': 'Michelle, cuaderno 22',
    'agro_celda.csv': 'Daniel, cuaderno 30',
    'lluvia_celda.csv': 'Michelle, cuaderno 12',
    'suelo_celda.csv': 'Katalina, cuaderno 42',
    'quema_celda.csv': 'Jonatan, cuaderno 24',
}

nombres = {os.path.basename(r) for r in glob.glob(os.path.join(COMUNES, '*_celda.csv'))}
nombres |= {os.path.basename(r)[:-4] for r in glob.glob(os.path.join(COMUNES, '*_celda.csv.dvc'))}

unidas = []
for nombre in sorted(nombres):
    traer(f'datos/comunes/{nombre}')
    capa = pd.read_csv(os.path.join(COMUNES, nombre))
    if 'id_celda' not in capa.columns or not capa['id_celda'].is_unique:
        print(f'Se omite {nombre}: necesita la columna id_celda y una fila por celda')
        continue
    sin_grilla = (~capa['id_celda'].isin(tabla['id_celda'])).sum()
    etiqueta = nombre.replace('_celda.csv', '')
    capa = capa.drop(columns=[c for c in capa.columns if c in DE_LA_GRILLA and c != 'id_celda'])
    capa = capa.rename(columns={c: f'{c}_{etiqueta}' for c in capa.columns if c != 'id_celda' and c in tabla.columns})
    tabla = tabla.merge(capa, on='id_celda', how='left')
    unidas.append(nombre)
    print(f'{nombre}: {capa.shape[1] - 1} variables, {capa["id_celda"].isin(grilla["id_celda"]).sum():,} celdas'
          + (f', {sin_grilla:,} filas con id_celda fuera de la grilla' if sin_grilla else ''))

faltan = {n: quien for n, quien in ESPERADAS.items() if n not in unidas}
print('\nCapas que faltan:', ', '.join(f'{n} ({q})' for n, q in faltan.items()) if faltan else 'ninguna')

assert len(tabla) == len(grilla) and tabla['id_celda'].is_unique
vacios = tabla.isna().mean().mul(100).round(1)
print('\n% de celdas sin dato por columna:')
print(vacios[vacios > 0].to_string() if (vacios > 0).any() else 'ninguna columna tiene vacíos')

agro_celda.csv: 8 variables, 34,763 celdas
focos_celda.csv: 5 variables, 34,763 celdas
lluvia_celda.csv: 5 variables, 34,763 celdas
quema_celda.csv: 4 variables, 34,763 celdas
suelo_celda.csv: 13 variables, 34,763 celdas

Capas que faltan: ninguna

% de celdas sin dato por columna:
agro_pct_2000       0.4
agro_pct_2024       0.4
cambio_agro_pp      0.4
pct_ph_menor_4_5    0.8
pct_ph_4_5_a_5_0    0.8
pct_ph_5_1_a_5_5    0.8
pct_ph_mayor_5_5    0.8
ph_kriging          2.4
cos_kriging_gkg     2.4


In [3]:
SALIDA_GRILLA = os.path.join(COMUNES, 'grilla_2km_variables.csv')
tabla.to_csv(SALIDA_GRILLA, index=False)
print(f'Guardado: {SALIDA_GRILLA} ({len(tabla):,} celdas, {tabla.shape[1]} columnas)')
tabla.head()

Guardado: c:\Users\mipip\Downloads\altillanura_proyecto_\altillanura_proyecto\datos\comunes\grilla_2km_variables.csv (34,763 celdas, 42 columnas)


,id_celda,mpio_cdpmp,mpio_cnmbr,area_km2,frac_celda,x_9377,y_9377,agro_pct_2000,agro_pct_2024,cambio_agro_pp,...,cos_medio_gkg,log_cos_medio,pct_area_ph_menor_5,clase_ph,pct_ph_menor_4_5,pct_ph_4_5_a_5_0,pct_ph_5_1_a_5_5,pct_ph_mayor_5_5,ph_kriging,cos_kriging_gkg
0,x4970_y2012,50573,PUERTO LÓPEZ,0.143016,0.035754,4971825.9,2013817.4,100.0000,87.3494,-12.6506,...,23.5111,3.1575,0.0000,fuertemente acido (5.1-5.5),0.0,66.6667,33.3333,0.0,NaN,NaN
1,x4970_y2014,50573,PUERTO LÓPEZ,0.451955,0.112989,4971653.6,2014335.3,71.6000,55.2000,-16.4000,...,31.3633,3.4456,0.0000,fuertemente acido (5.1-5.5),0.0,0.0000,100.0000,0.0,NaN,NaN
2,x4972_y2010,50573,PUERTO LÓPEZ,0.198384,0.049596,4973795.0,2011785.0,98.5981,100.0000,1.4019,...,22.6500,3.1202,0.0000,fuertemente acido (5.1-5.5),0.0,0.0000,100.0000,0.0,NaN,NaN
3,x4972_y2012,50573,PUERTO LÓPEZ,2.994346,0.748586,4973180.8,2013173.3,86.1176,89.2941,3.1765,...,22.4500,3.1113,9.7561,fuertemente acido (5.1-5.5),0.0,19.5122,80.4878,0.0,5.123,20.8249
4,x4972_y2014,50573,PUERTO LÓPEZ,1.611103,0.402776,4973102.6,2014418.3,84.9607,72.1100,-12.8507,...,25.6343,3.2439,5.8824,fuertemente acido (5.1-5.5),0.0,5.8824,94.1176,0.0,NaN,NaN


## Paso 3. Panel municipio × año
- **MapBiomas** (tablas municipales): área agropecuaria (nivel 1 "Zona agrícola y ganadera"), sabana (praderas y herbáceas) y palma, en hectáreas y en % del área total. `d_agro_pct` es el cambio neto agropecuario sobre el área total del año anterior, en puntos porcentuales, igual que en el panel de Jonatan.
- **EVA**: área sembrada total y de palma, soya, maíz y arroz. Un cultivo sin fila en un año que el municipio sí reportó cuenta como 0.
- **FIRMS**: focos del año y de la temporada seca.
- **CHIRPS**: lluvia del año y de la temporada seca, y sus anomalías frente a la normal 1991–2020.
- **ONI**: promedio del año y de la temporada seca. Es el mismo para los siete municipios.

In [4]:
municipios = pd.read_csv(os.path.join(COMUNES, 'altillanura_municipios.csv'), dtype={'mpio_cdpmp': str})
municipios = municipios[['mpio_cdpmp', 'mpio_cnmbr', 'area_ha']].rename(columns={'area_ha': 'area_mpio_ha'})

partes = []
for ruta in sorted(glob.glob(os.path.join(EXPANSION, 'datos', 'mapbiomas_crudos', '*.csv'))):
    mb = pd.read_csv(ruta, encoding='utf-8-sig')
    mb['Level 2'] = mb['Level 2'].fillna('').str.strip()
    mb = mb[mb['Level 2'] != '']
    largo = mb.melt(id_vars=['Level 1', 'Level 2'], var_name='anio', value_name='ha')
    largo['anio'] = largo['anio'].astype(int)
    largo['ha'] = pd.to_numeric(largo['ha'], errors='coerce').fillna(0.0)
    largo['mpio_cdpmp'] = re.search(r'(\d{5})_', os.path.basename(ruta)).group(1)
    partes.append(largo)
mb = pd.concat(partes, ignore_index=True)

claves = ['mpio_cdpmp', 'anio']
def suma(filtro):
    return mb[filtro].groupby(claves)['ha'].sum()

mapbiomas = pd.DataFrame({
    'area_total_ha': mb.groupby(claves)['ha'].sum(),
    'agro_ha': suma(mb['Level 1'] == 'Agricultural and livestock area'),
    'sabana_ha': suma(mb['Level 2'] == 'Grasslands / herbaceous'),
    'palma_mapbiomas_ha': suma(mb['Level 2'] == 'Palm oil'),
}).fillna(0.0).reset_index()
mapbiomas['agro_pct'] = mapbiomas['agro_ha'] / mapbiomas['area_total_ha'] * 100
mapbiomas['sabana_pct'] = mapbiomas['sabana_ha'] / mapbiomas['area_total_ha'] * 100
mapbiomas = mapbiomas.sort_values(claves)
previo = mapbiomas.groupby('mpio_cdpmp')[['agro_ha', 'area_total_ha']].shift(1)
mapbiomas['d_agro_pct'] = (mapbiomas['agro_ha'] - previo['agro_ha']) / previo['area_total_ha'] * 100

eva = pd.read_csv(os.path.join(EXPANSION, 'datos', 'eva', 'eva_altillanura_anual.csv'), encoding='utf-8-sig')
eva['mpio_cdpmp'] = eva['cod_mun'].astype(str).str.zfill(5)
CULTIVOS = {'PALMA DE ACEITE': 'eva_palma_ha', 'SOYA': 'eva_soya_ha', 'MAIZ': 'eva_maiz_ha', 'ARROZ': 'eva_arroz_ha'}
cultivos = eva.pivot_table(index=claves, columns='cultivo', values='area_sembrada_ha', aggfunc='sum', fill_value=0.0)
eva_panel = pd.DataFrame({'eva_sembrada_ha': eva.groupby(claves)['area_sembrada_ha'].sum()})
for cultivo, columna in CULTIVOS.items():
    eva_panel[columna] = cultivos[cultivo] if cultivo in cultivos.columns else 0.0
eva_panel = eva_panel.reset_index()

def mensual(df, columna_mes):
    df = df.copy()
    fecha = pd.to_datetime(df[columna_mes].astype(str).str[:7], format='%Y-%m')
    df['anio'] = fecha.dt.year
    df['m'] = fecha.dt.month
    df['temporada'] = np.where(df['m'] == 12, df['anio'] + 1, df['anio'])
    df['seca'] = df['m'].isin([12, 1, 2, 3])
    return df

def anual(df, grupo, columna, nombre, como='sum'):
    completo = df.groupby(grupo)['m'].nunique() == 12
    valor = df.groupby(grupo)[columna].agg(como)
    return valor[completo].rename(nombre)

def de_temporada(df, grupo, columna, nombre, como='sum'):
    seca = df[df['seca']]
    completo = seca.groupby(grupo)['m'].nunique() == 4
    valor = seca.groupby(grupo)[columna].agg(como)
    return valor[completo].rename(nombre).rename_axis(index={'temporada': 'anio'})

firms = mensual(pd.read_csv(os.path.join(COMUNES, 'firms_mensual_municipio.csv'), dtype={'mpio_cdpmp': str}), 'mes')
firms = firms[firms['anio'] >= 2003]
fuego = pd.concat([
    anual(firms, ['mpio_cdpmp', 'anio'], 'focos_total', 'focos_anual_total'),
    de_temporada(firms, ['mpio_cdpmp', 'temporada'], 'focos_total', 'focos_seca_total'),
], axis=1).reset_index()

chirps = mensual(pd.read_csv(os.path.join(COMUNES, 'chirps_mensual_municipio.csv'), dtype={'mpio_cdpmp': str}), 'mes')
lluvia = pd.concat([
    anual(chirps, ['mpio_cdpmp', 'anio'], 'lluvia_mm', 'lluvia_anual_mm'),
    anual(chirps, ['mpio_cdpmp', 'anio'], 'normal_mm', 'normal_anual_mm'),
    de_temporada(chirps, ['mpio_cdpmp', 'temporada'], 'lluvia_mm', 'lluvia_seca_mm'),
    de_temporada(chirps, ['mpio_cdpmp', 'temporada'], 'normal_mm', 'normal_seca_mm'),
], axis=1).reset_index()
lluvia['lluvia_anual_anom_pct'] = (lluvia['lluvia_anual_mm'] / lluvia['normal_anual_mm'] - 1) * 100
lluvia['lluvia_seca_anom_pct'] = (lluvia['lluvia_seca_mm'] / lluvia['normal_seca_mm'] - 1) * 100
lluvia = lluvia.drop(columns=['normal_anual_mm', 'normal_seca_mm'])

oni = mensual(pd.read_csv(os.path.join(COMUNES, 'oni_mensual.csv')), 'fecha')
enso = pd.concat([
    anual(oni, ['anio'], 'oni', 'oni_anual', 'mean'),
    de_temporada(oni, ['temporada'], 'oni', 'oni_seca', 'mean'),
], axis=1).reset_index()

anios = range(1985, 2026)
panel = pd.MultiIndex.from_product([municipios['mpio_cdpmp'], anios], names=claves).to_frame(index=False)
panel = panel.merge(municipios, on='mpio_cdpmp', how='left')
for parte in (mapbiomas, eva_panel, fuego, lluvia):
    panel = panel.merge(parte, on=claves, how='left')
panel = panel.merge(enso, on='anio', how='left').sort_values(claves).reset_index(drop=True)

assert panel.set_index(claves).index.is_unique and panel['mpio_cdpmp'].nunique() == 7
assert panel['agro_pct'].dropna().between(0, 100).all()
region = panel.groupby('anio')[['agro_ha', 'area_total_ha']].sum()
print('Altillanura, % agropecuario:', ' | '.join(f'{a}: {region.loc[a, "agro_ha"] / region.loc[a, "area_total_ha"] * 100:.1f} %' for a in (1985, 2000, 2024)))
print('\nAños con dato por variable:')
print(panel.drop(columns=['mpio_cdpmp', 'mpio_cnmbr', 'area_mpio_ha']).groupby('anio').count().gt(0).pipe(
    lambda t: t.apply(lambda c: f'{c[c].index.min()}–{c[c].index.max()}' if c.any() else 'sin dato')).to_string())

Altillanura, % agropecuario: 1985: 3.9 % | 2000: 6.7 % | 2024: 12.9 %

Años con dato por variable:
area_total_ha            1985–2024
agro_ha                  1985–2024
sabana_ha                1985–2024
palma_mapbiomas_ha       1985–2024
agro_pct                 1985–2024
sabana_pct               1985–2024
d_agro_pct               1986–2024
eva_sembrada_ha          2006–2025
eva_palma_ha             2006–2025
eva_soya_ha              2006–2025
eva_maiz_ha              2006–2025
eva_arroz_ha             2006–2025
focos_anual_total        2003–2025
focos_seca_total         2004–2025
lluvia_anual_mm          1985–2025
lluvia_seca_mm           1985–2025
lluvia_anual_anom_pct    1985–2025
lluvia_seca_anom_pct     1985–2025
oni_anual                1985–2025
oni_seca                 1985–2025


## Paso 4. Panel municipio × mes
Une los focos, la lluvia y el ONI mensuales. Los focos Terra + Aqua van desde enero de 2003; `focos_terra` está desde noviembre de 2000.

In [5]:
firms_mes = pd.read_csv(os.path.join(COMUNES, 'firms_mensual_municipio.csv'), dtype={'mpio_cdpmp': str})
firms_mes['focos_total'] = firms_mes['focos_total'].where(firms_mes['mes'] >= '2003-01')
chirps_mes = pd.read_csv(os.path.join(COMUNES, 'chirps_mensual_municipio.csv'), dtype={'mpio_cdpmp': str})
oni_mes = pd.read_csv(os.path.join(COMUNES, 'oni_mensual.csv'))
oni_mes['mes'] = oni_mes['fecha'].astype(str).str[:7]

panel_mes = (chirps_mes[['mpio_cdpmp', 'mes', 'lluvia_mm', 'normal_mm', 'anom_mm', 'anom_pct']]
             .merge(firms_mes[['mpio_cdpmp', 'mes', 'focos_total', 'focos_terra']], on=['mpio_cdpmp', 'mes'], how='outer')
             .merge(oni_mes[['mes', 'oni', 'fase']], on='mes', how='left')
             .merge(municipios[['mpio_cdpmp', 'mpio_cnmbr']], on='mpio_cdpmp', how='left'))
panel_mes = panel_mes[['mpio_cdpmp', 'mpio_cnmbr', 'mes', 'lluvia_mm', 'normal_mm', 'anom_mm', 'anom_pct',
                       'focos_total', 'focos_terra', 'oni', 'fase']].sort_values(['mpio_cdpmp', 'mes']).reset_index(drop=True)

assert panel_mes.set_index(['mpio_cdpmp', 'mes']).index.is_unique
print(f'Panel mensual: {len(panel_mes):,} filas, de {panel_mes["mes"].min()} a {panel_mes["mes"].max()}')

Panel mensual: 3,780 filas, de 1981-01 a 2025-12


## Paso 5. Guardar los paneles

In [6]:
SALIDA_ANIO = os.path.join(COMUNES, 'panel_municipio_anio.csv')
SALIDA_MES = os.path.join(COMUNES, 'panel_municipio_mes.csv')
panel.round(4).to_csv(SALIDA_ANIO, index=False)
panel_mes.round(4).to_csv(SALIDA_MES, index=False)

for ruta, original, llave in ((SALIDA_ANIO, panel, claves), (SALIDA_MES, panel_mes, ['mpio_cdpmp', 'mes'])):
    leida = pd.read_csv(ruta, dtype={'mpio_cdpmp': str})
    assert len(leida) == len(original) and leida.set_index(llave).index.is_unique
    print(f'Guardado: {ruta} ({len(leida):,} filas, {leida.shape[1]} columnas)')
panel[panel['anio'].isin([2000, 2024])].round(1)

Guardado: c:\Users\mipip\Downloads\altillanura_proyecto_\altillanura_proyecto\datos\comunes\panel_municipio_anio.csv (287 filas, 24 columnas)
Guardado: c:\Users\mipip\Downloads\altillanura_proyecto_\altillanura_proyecto\datos\comunes\panel_municipio_mes.csv (3,780 filas, 11 columnas)


,mpio_cdpmp,anio,mpio_cnmbr,area_mpio_ha,area_total_ha,agro_ha,sabana_ha,palma_mapbiomas_ha,agro_pct,sabana_pct,...,eva_maiz_ha,eva_arroz_ha,focos_anual_total,focos_seca_total,lluvia_anual_mm,lluvia_seca_mm,lluvia_anual_anom_pct,lluvia_seca_anom_pct,oni_anual,oni_seca
15,50325,2000,MAPIRIPÁN,1191747.9,1193129.1,115742.4,370767.2,0.0,9.7,31.1,...,NaN,NaN,NaN,NaN,3072.8,372.3,10.7,-7.9,-0.8,-1.3
39,50325,2024,MAPIRIPÁN,1191747.9,1193129.1,241215.2,303673.9,10465.3,20.2,25.5,...,200.0,50.0,893.0,673.0,2640.8,445.2,-4.8,10.2,0.4,1.6
56,50568,2000,PUERTO GAITÁN,1720078.4,1721864.7,306065.1,926892.9,852.6,17.8,53.8,...,NaN,NaN,NaN,NaN,2707.4,289.5,4.5,6.5,-0.8,-1.3
80,50568,2024,PUERTO GAITÁN,1720078.4,1721864.7,495551.2,777655.8,13514.0,28.8,45.2,...,44800.0,6100.0,1775.0,1258.0,2780.0,409.3,7.3,50.6,0.4,1.6
97,50573,2000,PUERTO LÓPEZ,689823.6,690884.1,267046.5,268014.1,882.7,38.7,38.8,...,NaN,NaN,NaN,NaN,2517.1,291.1,-7.4,-3.0,-0.8,-1.3
121,50573,2024,PUERTO LÓPEZ,689823.6,690884.1,332837.8,206227.1,2845.5,48.2,29.8,...,9915.0,17934.0,299.0,184.0,2910.3,505.9,7.1,68.5,0.4,1.6
138,99001,2000,PUERTO CARREÑO,1214232.3,1178791.9,34348.6,664570.1,0.0,2.9,56.4,...,NaN,NaN,NaN,NaN,2467.6,132.8,0.7,-3.0,-0.8,-1.3
162,99001,2024,PUERTO CARREÑO,1214232.3,1178791.9,142366.7,590298.9,0.0,12.1,50.1,...,73.0,0.0,731.0,589.0,2450.9,136.1,-0.0,-0.6,0.4,1.6
179,99524,2000,LA PRIMAVERA,1849210.7,1831799.1,30907.6,1272794.8,16.8,1.7,69.5,...,NaN,NaN,NaN,NaN,2795.6,177.1,5.4,-10.0,-0.8,-1.3
203,99524,2024,LA PRIMAVERA,1849210.7,1831799.1,218830.8,1101577.0,1024.4,11.9,60.1,...,100.0,1200.0,1385.0,781.0,2521.3,207.0,-4.9,5.2,0.4,1.6
